In [1]:
import numpy as np
import matplotlib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import random
from dataclasses import dataclass
from typing import Dict, Callable, Tuple, Optional

from symdisc import generate_euclidean_killing_fields_with_names
from symdisc.discovery import make_model_jacobian_callable_torch
from symdisc.enforcement.regularization.penalties import forward_with_invariance_penalty, forward_with_equivariance_penalty
from symdisc.enforcement.regularization.schedules import jump

In [2]:
# -------------------------
# Reproducibility
# -------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(1337)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# -------------------------
# Synthetic dataset
# -------------------------
class XYRDataset(Dataset):
    """
    Points in R^2 with target t = (1+x^2+y^2)*[x,y].
    A split can be made via the half-plane y >= 0 (train) vs y < 0 (test).
    """
    def __init__(self, X: np.ndarray, y: np.ndarray):
        assert X.shape[1] == 2
        assert y.ndim == 2 or (y.ndim == 2 and y.shape[1] == 2)
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float() #torch.from_numpy(y.reshape(-1, 1)).float()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def generate_points(n_total: int = 1000,
                    xy_range: float = 1.5) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate points uniformly in x,y ∈ [-xy_range, xy_range]
    Target t = (1+x^2+y^2)*[x,y].
    """
    X = np.empty((n_total, 2), dtype=np.float32)
    for i in range(n_total):
        x = np.random.uniform(-xy_range, xy_range)
        y = np.random.uniform(-xy_range, xy_range)
        X[i] = (x, y)
    y = X * (1 + X[:, 0]**2 + X[:, 1]**2)[:, None] #np.exp(X[:, 0]**2 + X[:, 1]**2) # + X[:, 2]**2)
    return X, y


def split_upper_lower_half_plane(X: np.ndarray, y: np.ndarray, upper_ratio=0.5):
    """
    Split the dataset by half-plane:
      - Train: y >= 0 (upper half-plane)
      - Test:  y < 0 (lower half-plane)
    If proportions are imbalanced, we still just use the predicate split.
    """
    upper_mask = X[:, 1] >= 0.0
    lower_mask = ~upper_mask

    X_train, y_train = X[upper_mask], y[upper_mask]
    X_test, y_test = X[lower_mask], y[lower_mask]

    # If either split is empty (unlikely), fallback to random split (just in case)
    if len(X_train) == 0 or len(X_test) == 0:
        n = len(X)
        idx = np.random.permutation(n)
        split = int(n * upper_ratio)
        train_idx, test_idx = idx[:split], idx[split:]
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

    return X_train, y_train, X_test, y_test

In [4]:
# -------------------------
# Small MLP regressor
# -------------------------
class SmallRegressor(nn.Module):
    def __init__(self, hidden=64, act="silu"):
        super().__init__()
        act_layer = nn.SiLU() if act == "silu" else nn.GELU()
        self.net = nn.Sequential(
            nn.Linear(2, hidden),
            act_layer,
            nn.Linear(hidden, hidden),
            act_layer,
            nn.Linear(hidden, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [5]:
# -------------------------
# Training utilities
# -------------------------
@dataclass
class TrainConfig:
    batch_size: int = 128
    epochs: int = 500
    lr: float = 1e-3
    weight_decay: float = 0.0
    lambda_R01: float = 1.0        # weight for rotation invariance penalty
    lambda_T2: float = 1.0         # initially off; set >0 to enforce z-translation too
    gamma_val: float = 0.5             # this should be strictly between 0 and 1
    gamma_wait: int = epochs//2
    print_every: int = 50


def evaluate(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    sse = 0.0
    mae = 0.0
    sum_y = 0.0
    sum_y2 = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            yhat = model(xb)

            sse += F.mse_loss(yhat, yb, reduction="sum").item()
            mae += F.l1_loss(yhat, yb, reduction="sum").item()

            sum_y  += float(yb.sum())
            sum_y2 += float((yb**2).sum())
            n += yb.numel()

    mse = sse / n
    mae = mae / n
    # SST relative to the split mean
    ybar = sum_y / n
    sst = max(sum_y2 - n * (ybar ** 2), 1e-12)  # guard against degenerate SST
    r2 = 1.0 - (sse / sst) if sst > 0 else float("nan")

    return {
        "MSE": mse,
        "MAE": mae,
        "R2": r2,
    }

In [6]:
X, y = generate_points(n_total=4000, xy_range=1.5)
X_train, y_train, X_test, y_test = split_upper_lower_half_plane(X, y)

# Euclidean Killing fields in ambient R^3
kvs, names = generate_euclidean_killing_fields_with_names(d=X.shape[1])

# Dataloaders
train_ds = XYRDataset(X_train, y_train)
test_ds = XYRDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, drop_last=False)

# Model + config
model = SmallRegressor(hidden=32, act="silu").to(device)
cfg = TrainConfig(
    batch_size=128,
    epochs=400,
    lr=1e-3,
    weight_decay=1e-4,
    lambda_R01=1.0,
    lambda_T2=0.0,
    print_every=50,
    gamma_val=0.5,   # between 0 and 1.
    gamma_wait=100
)

opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
model_criterion = nn.MSELoss()

Rxy = kvs[2]

active_fields = [Rxy]
weights = [1.0]

penalties_scaled = False

gamma_schedule = jump(cfg.gamma_val, cfg.gamma_wait)

In [7]:
for epoch in range(1, cfg.epochs + 1):
    model.train()
    running_mse = 0.0
    running_total = 0.0
    n_obs = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        gamma = float(gamma_schedule(epoch))

        if gamma!=0.0:
            yhat, sym_pen = forward_with_equivariance_penalty(
                model=model,
                X_in=active_fields,
                Y_out=active_fields,
                x=xb,
                loss=torch.nn.MSELoss(),
                weights=weights
            )
            #yhat, sym_pen = forward_with_invariance_penalty(
            #    model=model,
            #    X=active_fields,
            #    x=xb,
            #    loss=torch.nn.MSELoss(), #torch.nn.L1Loss(), #
            #    weights=weights
            #)
        else:
            yhat, sym_pen = model(xb), torch.tensor(0.0)

        model_loss = model_criterion(yhat, yb)

        if not penalties_scaled and gamma!=0.0:
            scale = model_loss.detach()/torch.max(sym_pen.detach(), torch.tensor(1e-8))
            penalties_scaled=True
        else:
            scale = 1.0

        loss = (1-gamma)*model_loss + gamma*scale*sym_pen

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        running_mse += float(model_loss.detach()) * yb.size(0)
        running_total += float(loss.detach()) * yb.size(0)
        n_obs += yb.size(0)


    if epoch % cfg.print_every == 0 or epoch == 1 or epoch == cfg.epochs:
        print("Model loss: " , (1-gamma)*model_loss.detach())
        print("Symmetry loss: ", gamma*scale*sym_pen.detach())
        train_metrics = evaluate(model, train_loader)
        test_metrics = evaluate(model, test_loader)
        print(f"[{epoch:03d}/{cfg.epochs}] "
              f"Train MSE: {train_metrics['MSE']:.4f}, R2: {train_metrics['R2']:.4f} | "
              f"Test MSE: {test_metrics['MSE']:.4f}, R2: {test_metrics['R2']:.4f} | "
              f"λ_R01={cfg.lambda_R01:.2f}")

Model loss:  tensor(6.8342)
Symmetry loss:  tensor(0.)
[001/400] Train MSE: 6.8949, R2: -0.0406 | Test MSE: 7.8838, R2: -0.1799 | λ_R01=1.00
Model loss:  tensor(0.0969)
Symmetry loss:  tensor(0.)
[050/400] Train MSE: 0.0887, R2: 0.9866 | Test MSE: 3.5711, R2: 0.4655 | λ_R01=1.00
Model loss:  tensor(0.0204)
Symmetry loss:  tensor(0.0685)
[100/400] Train MSE: 0.0258, R2: 0.9961 | Test MSE: 3.4412, R2: 0.4850 | λ_R01=1.00
Model loss:  tensor(0.0012)
Symmetry loss:  tensor(0.0015)
[150/400] Train MSE: 0.0036, R2: 0.9995 | Test MSE: 0.7212, R2: 0.8921 | λ_R01=1.00
Model loss:  tensor(0.0011)
Symmetry loss:  tensor(0.0006)
[200/400] Train MSE: 0.0019, R2: 0.9997 | Test MSE: 0.3999, R2: 0.9401 | λ_R01=1.00
Model loss:  tensor(0.0004)
Symmetry loss:  tensor(0.0009)
[250/400] Train MSE: 0.0011, R2: 0.9998 | Test MSE: 0.3361, R2: 0.9497 | λ_R01=1.00
Model loss:  tensor(0.0005)
Symmetry loss:  tensor(0.0002)
[300/400] Train MSE: 0.0008, R2: 0.9999 | Test MSE: 0.2914, R2: 0.9564 | λ_R01=1.00
Model